In [2]:
import pandas as pd
import numpy as np

# 读取Excel文件
df = pd.read_excel('D:/study/Media data proc/起点中文网月票榜.xlsx')

# 1. 查看数据基本信息
print("=== 数据基本信息 ===")
print(f"数据形状: {df.shape} (行数, 列数)")
print(f"\n列名: {list(df.columns)}")
print(f"\n数据类型:")
print(df.dtypes)

# 2. 查看前5行数据
print("\n=== 前5行数据预览 ===")
print(df.head())

# 3. 查看缺失值情况
print("\n=== 缺失值统计 ===")
missing_info = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    '缺失数量': missing_info,
    '缺失比例(%)': missing_percent.round(2)
})
print(missing_df[missing_df['缺失数量'] > 0])

# 4. 查看每列的唯一值数量（判断是否为无用的常量列）
print("\n=== 每列唯一值数量 ===")
unique_counts = df.nunique()
unique_df = pd.DataFrame({
    '列名': unique_counts.index,
    '唯一值数量': unique_counts.values,
    '占总行数比例(%)': (unique_counts.values / len(df) * 100).round(2)
})
print(unique_df)

# 5. 查看数据描述性统计（数值型列）
print("\n=== 数值型列描述性统计 ===")
print(df.describe())

=== 数据基本信息 ===
数据形状: (500, 18) (行数, 列数)

列名: ['标题', '标题链接', '图片', 'Unnamed: 3', '名称_链接', '作者', '作者_链接', '类型', '类型_链接', '标签', '连载状态', '简介', 'update_链接', 'update', '时间', 'total', 'redbtn', 'bluebtn']

数据类型:
标题             object
标题链接           object
图片             object
Unnamed: 3    float64
名称_链接          object
作者             object
作者_链接          object
类型             object
类型_链接          object
标签             object
连载状态           object
简介             object
update_链接      object
update         object
时间             object
total          object
redbtn         object
bluebtn        object
dtype: object

=== 前5行数据预览 ===
          标题                                     标题链接  \
0        捞尸人  https://www.qidian.com/book/1041637443/   
1  苟在初圣魔门当人材  https://www.qidian.com/book/1043182343/   
2    没钱修什么仙？  https://www.qidian.com/book/1042256511/   
3       玄鉴仙族  https://www.qidian.com/book/1035420986/   
4        夜无疆  https://www.qidian.com/book/1040765595/   

                         

In [3]:
import pandas as pd
import numpy as np
import re

def clean_qidian_data(file_path, save_path='D:/study/Media data proc/起点中文网月票榜.xlsx'):
    """
    起点中文网月票榜数据清洗函数
    
    参数:
    file_path: 原始Excel文件路径
    save_path: 清洗后文件保存路径
    """
    
    # 1. 读取原始数据
    df = pd.read_excel(file_path)
    print(f"原始数据形状: {df.shape}")
    
    # 2. 1  删除空列和固定值列（直接删除无价值列）
    columns_to_drop = [
        'Unnamed: 3',    # 空列
        'redbtn',        # 固定值列
        'bluebtn',       # 固定值列
        '标题链接',      # 链接列
        '图片',          # 链接列
        '作者_链接',     # 链接列
        '类型_链接',     # 链接列
        'update_链接',   # 链接列
        'total',         # 格式异常列
        '名称_链接'      # 重复信息列
    ]
    df_clean = df.drop(columns=columns_to_drop, errors='ignore')
    print(f"删除无用列后形状: {df_clean.shape}")
    
    # 2.2 处理时间列（转换为标准日期格式）
    if '时间' in df_clean.columns:
        # 去除非日期字符，提取纯日期
        df_clean['时间'] = pd.to_datetime(
            df_clean['时间'].astype(str).str.extract(r'(\d{4}-\d{2}-\d{2})')[0],
            errors='coerce'
        )
    
    # 2.3 处理update列（提取章节信息，去除冗余文本）
    if 'update' in df_clean.columns:
        df_clean['update'] = df_clean['update'].astype(str).str.replace('最新更新', '').str.strip()
    
    # 2.4 处理缺失值（删除关键信息缺失的行）
    key_columns = ['标题', '作者', '类型', '连载状态']  # 关键信息列
    df_clean = df_clean.dropna(subset=key_columns, how='any')
    print(f"删除关键信息缺失行后形状: {df_clean.shape}")
    
    # 2.5 去除重复行（按标题去重，避免重复数据）
    df_clean = df_clean.drop_duplicates(subset=['标题'], keep='first')
    print(f"去除重复行后形状: {df_clean.shape}")
    
    # 3. 保存清洗后的数据
    df_clean.to_excel(save_path, index=False)
    print(f"\n清洗完成！文件已保存至: {save_path}")
    
    # 4. 返回清洗后的数据
    return df_clean

# ------------------- 执行清洗 -------------------
# 使用函数进行数据清洗
cleaned_df = clean_qidian_data('D:/study/Media data proc/起点中文网月票榜.xlsx')

# 查看清洗结果预览
print("\n=== 清洗后数据预览 ===")
print(f"最终数据形状: {cleaned_df.shape}")
print(f"\n保留的列: {list(cleaned_df.columns)}")
print("\n前3行数据:")
print(cleaned_df.head(3))

原始数据形状: (500, 18)
删除无用列后形状: (500, 8)
删除关键信息缺失行后形状: (499, 8)
去除重复行后形状: (499, 8)

清洗完成！文件已保存至: D:/study/Media data proc/起点中文网月票榜.xlsx

=== 清洗后数据预览 ===
最终数据形状: (499, 8)

保留的列: ['标题', '作者', '类型', '标签', '连载状态', '简介', 'update', '时间']

前3行数据:
          标题     作者  类型    标签 连载状态  \
0        捞尸人  纯洁滴小龙  都市  异术超能   连载   
1  苟在初圣魔门当人材  鹤守月满池  玄幻  东方玄幻   连载   
2    没钱修什么仙？    熊狼狗  仙侠  修真文明   连载   

                                                  简介  \
0                            人知鬼恐怖，鬼晓人心毒。这是一本传统灵异小说。   
1  吕阳穿越修仙界，却成了魔门初圣宗的弟子。幸得异宝【百世书】，死后可以重开一世，让一切从头再来...   
2  老者：“你想报仇？”少年：“我被强者反复侮辱，被师尊视为垃圾，我怎么可能不想报仇？”老者摸了...   

                     update         时间  
0                   第四百五十八章 2025-11-17  
1        第一千一百四十七章 挖洞天法的墙角！ 2025-11-17  
2  第724章 跪下（求月票，差400多字8000） 2025-11-17  
